[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TianshuangQiu/TorchCode/blob/master/solutions/44_ranks_within_groups_solution.ipynb)

# 🟡 Solution: Ranks Within Groups

**Primitive: `scatter_` (one-hot) + `cumsum` + `gather`**

**Reduction:** `ranks[i]` = number of elements **before** position `i` in the same group.

Key steps:
1. Map `group_ids` to contiguous indices with `unique(return_inverse=True)` — handles non-contiguous ids like `[100, 50, 100]`
2. Build one-hot matrix `(N, G)` via `scatter_` — row `i` has a `1` in column `mapped[i]`
3. `cumsum(dim=0)` gives a running count of how many times each group has appeared so far
4. `gather` at each row's own group column → the count up to and including row `i`
5. Subtract 1 for 0-indexed ranks

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass

In [ ]:
import torch

In [ ]:
# ✅ SOLUTION

def ranks_within_groups(group_ids: torch.Tensor) -> torch.Tensor:
    # primitive: scatter_ builds one-hot; cumsum gives running count; gather reads own-group count
    _, mapped = group_ids.unique(return_inverse=True)         # contiguous group indices (N,)
    N = len(mapped)
    G = int(mapped.max().item()) + 1
    one_hot = torch.zeros(N, G, dtype=torch.long, device=group_ids.device)
    one_hot.scatter_(1, mapped.unsqueeze(1), 1)               # (N, G) one-hot
    cumsum = one_hot.cumsum(0)                                 # (N, G) running count per group
    return cumsum.gather(1, mapped.unsqueeze(1)).squeeze(1) - 1  # 0-indexed rank

In [ ]:
# Verify
group_ids = torch.tensor([2, 0, 2, 1, 0, 2])
result = ranks_within_groups(group_ids)
print('group_ids:', group_ids.tolist())
print('ranks:    ', result.tolist())
print('expected: ', [0, 0, 1, 0, 1, 2])

# Non-contiguous ids
print('non-contiguous [100,100,50]:', ranks_within_groups(torch.tensor([100, 100, 50])).tolist())

In [ ]:
# Run judge
from torch_judge import check
check("ranks_within_groups")